# Sparse Tensors (sptensor)
```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

Sparse tensors store only their nonzero entries, in coordinate format: an array of subscripts and a corresponding array of values. The `sptensor` class provides the same kinds of operations as the dense `tensor` class while using memory proportional to the number of nonzeros. Here we explain the basics for creating and working with sparse tensors.

[API reference](../sptensor.rst)

## Creating instances

### Creating a `sptensor`
The `sptensor` class stores the data in coordinate format. A sparse `sptensor` can be created by passing in a list of subscripts and values. For example, here we pass in three subscripts and a scalar value. The resulting sparse `sptensor` has three nonzero entries, and the `shape` is the size of the largest subscript in each dimension.

In [ ]:
from __future__ import annotations

import numpy as np

import pyttb as ttb

In [ ]:
np.random.seed(0)
subs = np.array([[0, 0, 0], [0, 1, 0], [2, 3, 1]])  # Subscripts of the nonzeros.
vals = np.array([[1], [2], [3]])  # Vals is a column vector; values of the nonzeros.
X = ttb.sptensor.from_aggregator(subs, vals)  # Sparse tensor with 3 nonzeros.

In [ ]:
X = ttb.sptensor(subs, vals, (3, 5, 2))  # Or, specify the shape explicitly.
X

Values corresponding to repeated subscripts are summed.

In [ ]:
subs = np.array(
    [[0, 0, 0], [0, 0, 2], [2, 2, 2], [3, 3, 3], [0, 0, 0], [0, 0, 0]]
)  # (1,1,1) is repeated.
vals = np.array([2, 2, 2, 2, 2, 2])[:, None]  # Vals is a column vector.
X = ttb.sptensor.from_aggregator(subs, vals)
X

### Specifying the accumulation method for the constructor

In [ ]:
subs = np.array([[0, 0, 0], [0, 0, 2], [2, 2, 2], [3, 3, 3], [0, 0, 0], [0, 0, 0]])
vals = 2 * np.ones((6, 1))  # A column vector of 2s
shape = (4, 4, 4)
X = ttb.sptensor.from_aggregator(subs, vals, shape, np.max)  # Maximum element.
X

In [ ]:
myfun = myfun = lambda x: np.sum(x) / 3  # Total sum divided by three.
X = ttb.sptensor.from_aggregator(
    subs, vals, shape, myfun
)  # Custom accumulation function.
X

### Creating a one-dimensional `sptensor`

In [ ]:
X = ttb.sptensor.from_aggregator(np.array([[0], [2], [4]]), np.ones((3, 1)))
X

In [ ]:
np.random.seed(0)
X = ttb.sptenrand((50,), nonzeros=5)
X

### Creating an all-zero `sptensor`

In [ ]:
X = ttb.sptensor()
X[9, 9, 9] = 0  # Creates an all-zero tensor.
X

### Creating a `sptensor` from its constituent parts

In [ ]:
np.random.seed(0)
X = ttb.sptenrand([40, 30, 20], nonzeros=5)  # Create data.
Y = X.copy()
Y

### Creating an empty `sptensor`

In [ ]:
Y = ttb.sptensor()  # Create an empty sptensor.
Y

### Use `sptenrand` to create a random `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.sptenrand(
    [10, 10, 10], 0.01
)  # Create a tesnor with 1% nonzeros using the 'density' param.
X

In [ ]:
np.random.seed(0)
X = ttb.sptenrand(
    [10, 10, 10], nonzeros=10
)  # Create a tensor with 10 nonzeros using the 'nonzeros' param.
X

### Use `find` to extract nonzeros from a `tensor` and then create a `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.tensor(np.random.rand(5, 4, 2))  # Create a tensor.
larger_entries = X > 0.9  # Extract subscipts of values greater than 0.9.
subs, vals = larger_entries.find()  # Extract corresponding subscripts and values.
Y = ttb.sptensor.from_aggregator(subs, vals)  # Create a new sptensor.
Y

## Key properties

### Constituent parts of a `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.sptenrand([40, 30, 20], nonzeros=5)  # Create data.
X.subs  # Subscripts of nonzeros.

In [ ]:
X.vals  # Corresponding nonzero values.

In [ ]:
X.shape  # The shape.

### Use `ndims` and `shape` to get the shape of a `sptensor`

In [ ]:
X = ttb.sptensor(
    np.array([[1, 1, 1], [2, 3, 2], [3, 4, 1], [1, 0, 0]]),
    np.array([[3], [2], [1], [3]]),
    (4, 5, 3),
)
X

In [ ]:
X.ndims  # Number of dimensions or modes.

In [ ]:
X.shape  # Shape of X.

In [ ]:
X.shape[2]  # Shape of mode 3 of X.

### Use `nnz` to get the number of nonzeroes of a `sptensor`

In [ ]:
X = ttb.sptensor(
    np.array([[1, 1, 1], [2, 3, 2], [3, 4, 1], [1, 0, 0]]),
    np.array([[3], [2], [1], [3]]),
    (4, 5, 3),
)
X.nnz  # Number of nonzeros of X.

## Converting to other types

### Use `full` or `to_tensor` to convert a `sptensor` to a (dense) `tensor`

In [ ]:
indices = np.array([[0, 0, 0], [1, 1, 1]])
values = np.array([[1], [1]])
X = ttb.sptensor.from_aggregator(indices, values)  # Create a sparse tensor.
X.full()  # Convert it to a (dense) tensor.

In [ ]:
Y = X.to_tensor()  # Same as above.
Y

### Use `to_sptensor` to convert a (dense) `tensor` to a `sptensor`

In [ ]:
indices = np.array([[0, 0, 0], [1, 1, 1]])
values = np.array([[1], [1]])
X = ttb.sptensor.from_aggregator(indices, values)  # Create a sparse tensor.
Y = X.to_tensor()  # Convert it to a (dense) tensor.
Z = Y.to_sptensor()  # Convert a tensor to a sptensor.
Z

### Use `double` to convert a `sptensor` to a (dense) multidimensional array

In [ ]:
indices = np.array([[0, 0, 0], [1, 1, 1]])
values = np.array([[1], [1]])
X = ttb.sptensor.from_aggregator(indices, values)  # Create a sparse tensor.
Y = ttb.sptensor.double(X)  # Creates numpy.ndarray
Y

## Core methods

### Use `squeeze` to remove singleton dimensions from a `sptensor`

In [ ]:
indices = np.array([[0, 0, 0], [1, 0, 0]])
values = np.ones((2, 1))

Y = ttb.sptensor.from_aggregator(indices, values)  # Create a sparse tensor.
Y

In [ ]:
Y.squeeze()  # Remove singleton dimensions.

### Use `squash` to remove empty slices from a `sptensor`

In [ ]:
indices = np.array([[0, 0, 0], [2, 2, 2]])
values = np.array([[1], [3]])

Y = ttb.sptensor.from_aggregator(indices, values)  # Create a sparse tensor.
Y

In [ ]:
Y.squash()

### Subscripted reference for a `sptensor`

In [ ]:
X = ttb.sptensor(
    np.array([[3, 3, 3], [1, 1, 0], [1, 2, 1]]), np.array([[3], [5], [1]]), (4, 4, 4)
)  # Create a sptensor.
X

In [ ]:
X[0, 1, 0]  # Extract the (0,1,0) element, which is zero.

In [ ]:
X[3, 3, 3]  # Extract the (3,3,3) element, which is non-zero.

In [ ]:
X[0:2, 1:4, :]  # Extract the 2x3x4 subtensor.

In [ ]:
X[1, 1, 1]

In [ ]:
X[1, 1, 0]

In [ ]:
X[[0, 5]]  # Same as above but with linear indices.

In [ ]:
indices = np.array([[0], [2], [4]])
values = np.array([[1], [1], [1]])
X = ttb.sptensor.from_aggregator(indices, values)
X

In [ ]:
X[(2,)]

In [ ]:
X[[2, 4],]  # Returns a subtensor.

### Subscripted assignment for a `sptensor`

In [ ]:
X = ttb.sptensor(np.array([[]]), np.array([[]]), (30, 40, 20))
X

In [ ]:
X[29, 39, 19] = 7  # Assign a single element.
X

In [ ]:
X[0, 0, 0], X[1, 1, 1] = [1, 1]  # Assign a list of elements.
X

In [ ]:
np.random.seed(0)
Y = ttb.sptenrand((10, 10, 10), nonzeros=10)
X[10:20, 10:20, 10:20] = Y  # Assign a subtensor.

In [ ]:
X[30, 40, 20] = 4  # Grows the shape of the sptensor.
X

In [ ]:
X[110:120, 110:120, 110:120] = ttb.sptenrand((10, 10, 10), nonzeros=10)  # Grow more.

### Using negative indexing for the last array index

In [ ]:
X[-10:, -10:, -5:]

### Use `elemfun` to manipulate the nonzeros of a `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.sptenrand((10, 10, 10), nonzeros=3)
X

In [ ]:
Z = X.elemfun(lambda value: np.sqrt(value))  # Square root of every nonzero.
Z

In [ ]:
Z = X.elemfun(lambda value: value + 1)  # Use a custom function.
Z

In [ ]:
Z = X.ones()  # Change every nonzero to one.
Z

### Use `permute` to reorder the modes of a `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.sptenrand((30, 40, 20, 1), nonzeros=5)  # Create data.
X

In [ ]:
X.permute(np.array([3, 2, 1, 0]))  # Reorder the modes.

`permute` works correctly for a 1-dimensional `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.sptenrand((40,), nonzeros=4)  # Create data.
X

In [ ]:
X.permute(np.array([0]))

### Use `allsubs` to list every possible subscript of a `sptensor`
There is one row for each of the entries of the tensor, zero or not.

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
S_m.allsubs().shape

### Use `logical_and`, `logical_or`, `logical_xor`, and `logical_not`
The logical operators treat nonzeros as true. `logical_or` and `logical_xor` with a dense `tensor` return a `tensor`.

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
S2_m = ttb.sptensor(
    np.array([[0, 0, 0], [0, 1, 1]]), np.array([[1.0], [1.0]]), shape=(2, 3, 2)
)
print(S_m.logical_and(S2_m))
print(S_m.logical_or(S2_m))
print(S_m.logical_xor(S2_m))
print(S_m.logical_not().nnz)

### Use `mask` to extract the values of a `sptensor` at the nonzeros of a mask

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
W_m = ttb.sptensor(
    np.array([[1, 1, 1], [1, 2, 0]]), np.array([[1], [1]]), shape=(2, 3, 2)
)
S_m.mask(W_m)

### Use `norm` to compute the Frobenius norm of a `sptensor`

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
S_m.norm()

### Use `nvecs` to compute the leading mode-$n$ vectors of a `sptensor`

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
S_m.nvecs(0, 1)

### Use `reshape` to change the shape of a `sptensor`

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
S_m.reshape((3, 2, 2))

### Use `spmatrix` to convert a 2-way `sptensor` to a SciPy sparse matrix

In [ ]:
S2d_m = ttb.sptensor(
    np.array([[0, 0], [1, 1], [1, 2]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3),
)
M_sp = S2d_m.spmatrix()
print(type(M_sp))
M_sp.toarray()

### Use `subdims` to find the nonzeros within a region of a `sptensor`
The region has one entry per mode, given as an index or an array of indices. The result gives the locations (rows of `subs`) of the nonzeros inside the region.

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
S_m.subdims([1, np.array([1, 2]), np.array([0, 1])])

The methods `ttm`, `ttv`, `mttkrp`, `innerprod`, and `contract` are covered in the [tensor multiplication](utility_multiplication.ipynb) tutorial; `collapse` and `scale` in the [collapsing and scaling](utility_collapse_scale.ipynb) tutorial; `from_function` and `from_aggregator` in the [creating tensors](utility_tensor_creation.ipynb) tutorial; and `to_sptenmat` in the [sptenmat](class_sptenmat.ipynb) tutorial.

## Operators

### Basic operations (plus, minus, times, etc.) on a `sptensor`
`sptensor`s support plus, minus, times, divide, power, equals, and not-equals operators.  `sptensor`s can use their operators with another `sptensor` or a scalar (with the exception of equalities which only takes `sptensor`s).  All mathematical operators are elementwise operations.

#### Addition

In [ ]:
X = ttb.sptensor(np.array([[0, 0], [1, 1]]), np.array([[2], [2]]), (2, 2))
X

In [ ]:
Y = ttb.sptensor(np.array([[0, 0], [0, 1]]), np.array([[3], [3]]), (2, 2))
Y

In [ ]:
+X  # Calls uplus.

In [ ]:
X + 1  # This addition yields dense tensor

In [ ]:
X + Y  # This addition yields sparse tensor

In [ ]:
X += 2
X

#### Subtraction

In [ ]:
X = ttb.sptensor(np.array([[0, 0], [1, 1]]), np.array([[2], [2]]), (2, 2))
Y = ttb.sptensor(np.array([[0, 0], [0, 1]]), np.array([[3], [3]]), (2, 2))

In [ ]:
-X  # Calls uminus.

In [ ]:
X - Y  # Calls minus.

#### Multiplication

In [ ]:
X = ttb.sptensor(np.array([[0, 0], [1, 1]]), np.array([[2], [2]]), (2, 2))
Y = ttb.sptensor(np.array([[0, 0], [0, 1]]), np.array([[3], [3]]), (2, 2))

In [ ]:
X * Y  # Calls times.

In [ ]:
X * 5  # Calls mtimes.

#### Division

In [ ]:
X = ttb.sptensor(np.array([[0, 0], [1, 1]]), np.array([[2], [2]]), (2, 2))
Y = ttb.sptensor(np.array([[0, 0], [0, 1]]), np.array([[3], [3]]), (2, 2))

In [ ]:
X / 2  # Calls rdivide.

In [ ]:
X / Y  # Divide by Zero

In [ ]:
X /= 4
print(X)

#### Equality

In [ ]:
X = ttb.sptensor(np.array([[0, 0], [1, 1]]), np.array([[2], [2]]), (2, 2))
Y = ttb.sptensor(np.array([[0, 0], [0, 1]]), np.array([[3], [3]]), (2, 2))

In [ ]:
(X == Y)

In [ ]:
X.isequal(Y)

In [ ]:
X != Y

### Displaying a `sptensor`

In [ ]:
np.random.seed(0)
X = ttb.sptenrand((40,), nonzeros=4)  # Create data.
print(X)

In [ ]:
X  # In the python interface